# 02 — Score TMDB Movies with NRC VAD + Emotion Intensity

Loads the full TMDB dataset (~930K movies), pre-scores every unique keyword
using the pickled NRC lookup dicts from `01_lexicons.ipynb`, then aggregates
to movie level and exports to CSV for Kaggle upload.

**Prerequisites:** run `01_lexicons.ipynb` first to generate `data/lexicons/nrc_lookups.pkl`.

In [1]:
import os
import pickle
import re
import time
import zipfile
from pathlib import Path

import kagglehub
import pandas as pd
from dotenv import load_dotenv
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

load_dotenv()

DATASET  = 'asaniczka/tmdb-movies-dataset-2023-930k-movies'
NROWS    = None   # None = full ~930K dataset; set to 10_000 for quick tests

_lemmatizer = WordNetLemmatizer()

def _tokenize(keyword: str) -> list:
    return [t for t in re.split(r'[\s\-_/]+', keyword.lower()) if t.isalpha()]

def _lemma_candidates(tok: str) -> list:
    seen, out = set(), []
    for pos in ('n', 'v', 'a'):
        lemma = _lemmatizer.lemmatize(tok, pos=pos)
        if lemma not in seen:
            seen.add(lemma)
            out.append(lemma)
    if tok not in seen:
        out.append(tok)
    return out

In [2]:
# ── Load TMDB dataset ─────────────────────────────────────────────────────
t0 = time.time()
base = Path(kagglehub.dataset_download(DATASET))
csvs = list(base.rglob('*.csv')) or []
if not csvs:
    zips = sorted(base.rglob('*.zip'))
    import zipfile
    with zipfile.ZipFile(zips[0]) as z:
        csvs = [z.extract(n, base) for n in z.namelist() if n.endswith('.csv')]

pandas_kwargs = {'nrows': NROWS} if NROWS else {}
df = pd.read_csv(sorted(csvs)[0], **pandas_kwargs)
print(f'Loaded {len(df):,} rows in {time.time()-t0:.1f}s')

Loaded 1,382,594 rows in 5.9s


In [3]:
# ── Load pickled NRC lookup dicts ─────────────────────────────────────────
with open('data/lexicons/nrc_lookups.pkl', 'rb') as f:
    nrc = pickle.load(f)

intensity_lookup = nrc['intensity_lookup']
vad_lookup       = nrc['vad_lookup']
EMOTIONS         = nrc['EMOTIONS']
VAD_DIMS         = nrc['VAD_DIMS']

print(f'intensity_lookup: {len(intensity_lookup):,} words')
print(f'vad_lookup:       {len(vad_lookup):,} words')

intensity_lookup: 5,891 words
vad_lookup:       19,971 words


In [4]:
# ── Pre-score all unique keywords ─────────────────────────────────────────
# Do this ONCE across all unique keywords rather than per-movie.
# With 930K movies sharing ~150K unique keywords, this is a 6x speedup.
t0 = time.time()

unique_keywords = sorted(
    {kw.strip() for s in df['keywords'].dropna().astype(str) for kw in s.split(',') if kw.strip()}
)
print(f'Unique keywords: {len(unique_keywords):,}')


def _score_keyword(keyword, intensity_lookup, vad_lookup, emotions, vad_dims):
    tokens = _tokenize(keyword)
    e_rows, v_rows = [], []
    for tok in tokens:
        for lemma in _lemma_candidates(tok):
            if lemma in intensity_lookup:
                e_rows.append(intensity_lookup[lemma])
                break
        for lemma in _lemma_candidates(tok):
            if lemma in vad_lookup:
                v_rows.append(vad_lookup[lemma])
                break
    emotion_scores = (
        {e: sum(r[e] for r in e_rows) / len(e_rows) for e in emotions}
        if e_rows else {e: 0.0 for e in emotions}
    )
    vad_scores = (
        {d: sum(r[d] for r in v_rows) / len(v_rows) for d in vad_dims}
        if v_rows else {d: float('nan') for d in vad_dims}
    )
    return {**emotion_scores, **vad_scores}


kw_scores = {
    kw: _score_keyword(kw, intensity_lookup, vad_lookup, EMOTIONS, VAD_DIMS)
    for kw in unique_keywords
}
print(f'Pre-scored {len(kw_scores):,} keywords in {time.time()-t0:.1f}s')

Unique keywords: 67,908


Pre-scored 67,908 keywords in 2.8s


In [5]:
# ── Aggregate to movie level (optimised) ─────────────────────────────────
t0 = time.time()

def score_movie(keywords_str):
    if not isinstance(keywords_str, str) or not keywords_str.strip():
        return {**{e: 0.0 for e in EMOTIONS}, **{d: float('nan') for d in VAD_DIMS}}
    kws = [kw.strip() for kw in keywords_str.split(',') if kw.strip()]

    e_rows = [kw_scores[kw] for kw in kws if kw in kw_scores]
    emotion_means = {}
    for e in EMOTIONS:
        vals = [r[e] for r in e_rows if r[e] > 0]
        emotion_means[e] = sum(vals) / len(vals) if vals else 0.0

    vad_means = {}
    for d in VAD_DIMS:
        vals = [r[d] for r in e_rows if r[d] == r[d]]  # NaN-safe
        vad_means[d] = sum(vals) / len(vals) if vals else float('nan')

    return {**emotion_means, **vad_means}


# pd.DataFrame(tolist()) is ~10x faster than .apply(pd.Series) on large frames
scores_df = pd.DataFrame(df['keywords'].map(score_movie).tolist())
movies = pd.concat([df[['id', 'title', 'keywords']], scores_df], axis=1)

movies['sentiment'] = movies['valence'].apply(
    lambda v: 'positive' if v > 0.5 else ('negative' if v < 0.5 else 'neutral')
    if v == v else 'unknown'
)

elapsed = time.time() - t0
print(f'Scored {len(movies):,} movies in {elapsed:.1f}s ({len(movies)/elapsed:,.0f} movies/sec)')
print()
print('Sentiment distribution:')
print(movies['sentiment'].value_counts().to_string())

Scored 1,382,594 movies in 3.3s (414,070 movies/sec)

Sentiment distribution:
sentiment
unknown     1061699
positive     223080
negative      97153
neutral         662


In [ ]:
import tracemalloc
import gc

# ── Performance benchmark ─────────────────────────────────────────────────
print('=' * 55)
print('PERFORMANCE BENCHMARK')
print('=' * 55)

# Keyword pre-scoring
t_kw_start = time.perf_counter()
_ = {
    kw: _score_keyword(kw, intensity_lookup, vad_lookup, EMOTIONS, VAD_DIMS)
    for kw in unique_keywords
}
t_kw = time.perf_counter() - t_kw_start
print(f'Keyword scoring:   {len(unique_keywords):>8,} keywords  {t_kw:.2f}s  ({len(unique_keywords)/t_kw:,.0f} kw/sec)')

# Movie scoring (with memory tracking)
gc.collect()
tracemalloc.start()
t_mv_start = time.perf_counter()
scores_list = df['keywords'].map(score_movie).tolist()
t_mv = time.perf_counter() - t_mv_start
_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'Movie scoring:     {len(df):>8,} movies    {t_mv:.2f}s  ({len(df)/t_mv:,.0f} movies/sec)')
print(f'Peak memory:       {peak_bytes/1e6:>8.1f} MB')

# Coverage stats
scores_df_tmp = pd.DataFrame(scores_list)
has_valence  = scores_df_tmp['valence'].notna().sum()
has_emotion  = (scores_df_tmp[EMOTIONS].sum(axis=1) > 0).sum()
no_keywords  = df['keywords'].isna().sum()
print()
print('COVERAGE')
print('-' * 55)
print(f'Movies total:      {len(df):>8,}')
print(f'No keywords:       {no_keywords:>8,}  ({100*no_keywords/len(df):.1f}%)')
print(f'VAD valence match: {has_valence:>8,}  ({100*has_valence/len(df):.1f}%)')
print(f'Emotion signal:    {has_emotion:>8,}  ({100*has_emotion/len(df):.1f}%)')
print(f'Unique keywords:   {len(unique_keywords):>8,}')
print('=' * 55)
del scores_df_tmp, scores_list

In [6]:
# ── Sanity check ──────────────────────────────────────────────────────────
print('Most positive valence:')
display(movies.nlargest(5, 'valence')[['title', 'valence', 'arousal', 'dominance', 'sentiment', 'keywords']])
print()
print('Most negative valence:')
display(movies.nsmallest(5, 'valence')[['title', 'valence', 'arousal', 'dominance', 'sentiment', 'keywords']])

Most positive valence:


,title,valence,arousal,dominance,sentiment,keywords
5811,Little Italy,1.0,0.519,0.673,positive,love
5827,A Perfect Plan,1.0,0.519,0.673,positive,"france, love"
7641,Ma che bella sorpresa,1.0,0.519,0.673,positive,love
8069,Skin,1.0,0.519,0.673,positive,"skinhead, love"
8627,A Short Film About Love,1.0,0.519,0.673,positive,love



Most negative valence:


,title,valence,arousal,dominance,sentiment,keywords
82673,The Johnsons,0.005,0.81,0.436,negative,"nightmare, menstruation"
91267,Dream House Nightmare,0.005,0.81,0.436,negative,nightmare
100909,Shhh...,0.005,0.81,0.436,negative,nightmare
126530,Bed Time,0.005,0.81,0.436,negative,nightmare
154645,Ingenium,0.005,0.81,0.436,negative,nightmare


In [7]:
# ── Write output CSV ──────────────────────────────────────────────────────
import json

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'movie_vad_scores.csv'

out_cols = ['id', 'title', 'keywords', 'sentiment', 'valence', 'arousal', 'dominance'] + EMOTIONS
movies[out_cols].to_csv(OUTPUT_CSV, index=False)

size_mb = OUTPUT_CSV.stat().st_size / 1e6
print(f'Wrote {len(movies):,} rows → {OUTPUT_CSV}  ({size_mb:.1f} MB)')

# Dataset metadata for Kaggle
kaggle_username = os.environ.get('KAGGLE_USERNAME', '')
if not kaggle_username:
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        kaggle_username = json.loads(kj.read_text()).get('username', '')

DATASET_SLUG = 'tmdb-movie-vad-emotion-scores'
meta = {
    'title': 'TMDB Movie VAD + Emotion Intensity Scores',
    'id': f'{kaggle_username}/{DATASET_SLUG}' if kaggle_username else f'YOUR_USERNAME/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
    'description': (
        '~930K TMDB movies scored with NRC VAD (valence/arousal/dominance) '
        'and NRC Emotion Intensity (anger, fear, joy, sadness, disgust, '
        'surprise, anticipation, trust) derived from movie keywords. '
        'Valence > 0.5 = positive sentiment.'
    ),
}
(OUTPUT_DIR / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
print(f'Metadata: {OUTPUT_DIR}/dataset-metadata.json')
print(f'Dataset id: {meta["id"]}')

Wrote 1,382,594 rows → output/movie_vad_scores.csv  (128.3 MB)
Metadata: output/dataset-metadata.json
Dataset id: YOUR_USERNAME/tmdb-movie-vad-emotion-scores


In [8]:
# ── Upload to Kaggle ──────────────────────────────────────────────────────
# Uncomment and run after verifying the CSV looks correct.
#
# First time (creates new dataset):
#   !kaggle datasets create -p output
#
# Subsequent updates:
#   !kaggle datasets version -p output -m "Updated scores"
print('Ready to upload. Run one of the commands above.')

Ready to upload. Run one of the commands above.
